# שלב 3 — הדגמה על מפה (Colab Enterprise)

מציג על מפה אינטראקטיבית את **100 תתי-המסלולים הארוכים-פופולריים** לכל קונפיגורציה (≥1/3/5/10/20/40 ק"מ), עבור כל אחת מהשיטות. כולל:

1. מפה עם **שכבות נפרדות לכל ספֵּף מרחק** (בורר שכבות)
2. **מפת חום** של מוקדי פעילות (תאי H3 הכי עמוסים)
3. מפת **השוואה בין השיטות** על אותו אזור

> ב-Colab: התקינו `folium h3 pyspark` בתא הראשון. המפות מוצגות inline ונשמרות גם כקובצי HTML.


## 0. התקנה (Colab בלבד)

In [ ]:
# ב-Colab Enterprise הריצו פעם אחת:
# !pip install -q folium h3 pyspark


## 1. טעינת תוצאות השיטות

In [ ]:
import os, folium
from folium.plugins import HeatMap
import h3
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("porto-taxi-map").master("local[*]").config("spark.driver.memory","4g").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

USE_SAMPLE = True
SUFFIX = "_sample" if USE_SAMPLE else ""
RESULTS = {
    "graph_cms":      f"out/subroutes_graph{SUFFIX}.parquet",
    "suffix_hll":     f"out/subroutes_suffix{SUFFIX}.parquet",
    "clustering_lsh": f"out/subroutes_clustering{SUFFIX}.parquet",
}
ENCODED = f"out/trips_h3{SUFFIX}.parquet"   # למפת החום

CONFIGS_KM = [1, 3, 5, 10, 20, 40]
TOP_N = 100
PORTO = [41.1579, -8.6291]

results = {}
for m, path in RESULTS.items():
    if os.path.exists(path):
        results[m] = spark.read.parquet(path).select("win_key","support","km").toPandas()
print("loaded:", {m: len(df) for m, df in results.items()})


## 2. פונקציית מפה — Top-100 לכל קונפיגורציה (שכבות)

In [ ]:
COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]

def route_points(win_key):
    return [list(h3.cell_to_latlng(c)) for c in win_key.split(">")]

def map_per_config(pdf, title):
    m = folium.Map(location=PORTO, zoom_start=13, tiles="cartodbpositron")
    for K, col in zip(CONFIGS_KM, COLORS):
        sub = pdf[pdf.km >= K].sort_values("km", ascending=False).head(TOP_N)
        if len(sub) == 0:
            continue
        fg = folium.FeatureGroup(name=f">= {K} km  (top {len(sub)})", show=(K == CONFIGS_KM[0]))
        for _, r in sub.iterrows():
            folium.PolyLine(route_points(r.win_key), color=col, weight=3, opacity=0.6,
                            popup=f"len={r.km:.2f} km · support={int(r.support)}").add_to(fg)
        fg.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    title_html = f'<h4 style="position:fixed;top:10px;left:60px;z-index:9999;background:white;padding:4px">{title}</h4>'
    m.get_root().html.add_child(folium.Element(title_html))
    return m


## 3. מפה לשיטה הנבחרת

In [ ]:
METHOD = "graph_cms"     # graph_cms / suffix_hll / clustering_lsh
m1 = map_per_config(results[METHOD], f"Top-{TOP_N} popular sub-routes — {METHOD}")
m1.save(f"out/map_{METHOD}.html")
print("saved:", f"out/map_{METHOD}.html")
m1   # מוצג inline ב-Colab


## 4. מפת חום — מוקדי פעילות (תאי H3 הכי עמוסים)

In [ ]:
enc = spark.read.parquet(ENCODED).select("cells")
hot = (enc.select(F.explode("cells").alias("cell"))
          .groupBy("cell").count()
          .orderBy(F.desc("count")).limit(2000).toPandas())
heat = [[*h3.cell_to_latlng(c), int(w)] for c, w in zip(hot.cell, hot["count"])]

mh = folium.Map(location=PORTO, zoom_start=13, tiles="cartodbpositron")
HeatMap(heat, radius=12, blur=15).add_to(mh)
mh.save("out/map_hotspots.html")
print("saved: out/map_hotspots.html  (top cells:", len(heat), ")")
mh


## 5. השוואת שיטות על מפה אחת (Top-100 ≥1 ק"מ)

In [ ]:
mc = folium.Map(location=PORTO, zoom_start=13, tiles="cartodbpositron")
method_colors = {"graph_cms": "#e41a1c", "suffix_hll": "#377eb8", "clustering_lsh": "#4daf4a"}
K0 = CONFIGS_KM[0]
for method, pdf in results.items():
    sub = pdf[pdf.km >= K0].sort_values("km", ascending=False).head(TOP_N)
    fg = folium.FeatureGroup(name=f"{method} ({len(sub)})", show=True)
    for _, r in sub.iterrows():
        folium.PolyLine(route_points(r.win_key), color=method_colors.get(method, "#333"),
                        weight=2.5, opacity=0.5).add_to(fg)
    fg.add_to(mc)
folium.LayerControl(collapsed=False).add_to(mc)
mc.save("out/map_compare.html")
print("saved: out/map_compare.html")
mc


## הערות להדגמה
- כבו/הדליקו שכבות בבורר (ימין-עליון) כדי להציג קונפיגורציות שונות.
- על המדגם רק ≥1 ו-≥3 ק"מ מאוכלסים; על הדאטה המלא ב-DataProc יופיעו גם המסלולים הארוכים (≥5/10/20/40 ק"מ).
- במפת ההשוואה רואים ש-`graph_cms` ו-`suffix_hll` חופפים (אותם מסלולים), ו-`clustering_lsh` מכסה תת-קבוצה.
